<a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Proyecto Final: Análisis de Imágenes de Malezas — OPPD Dataset
### Big Data y Bioinformática
---
**Dataset:** Open Plant Phenotyping Database (OPPD) — Full Images  
**Artículo:** Madsen et al. (2020). *Open Plant Phenotype Database of Common Weeds in Denmark*. Remote Sensing, 12(8), 1246.  
**Licencia:** CC BY-NC-SA 4.0  
**Fuente:** https://vision.eng.au.dk/open-plant-phenotyping-database/

**Integrantes:** Mariana Castaño Giraldo, Linda Liceth Cadena Trujillo, Juan José Pineda Pedraza

---

## 📋 Descripción del Dataset

El OPPD (Open Plant Phenotyping Database) es una base de datos pública de imágenes RGB de alta resolución para la detección y clasificación visual de malezas. El dataset completo contiene 7,590 imágenes de 47 especies de malezas comunes en cultivos de Dinamarca, con 315,038 objetos anotados mediante bounding boxes (cajas delimitadoras).  


El subconjunto disponible para este trabajo contiene 98 imágenes de resolución 3000×4096 px con 97 archivos de anotación JSON (3,894 bounding boxes, 52 clases codificadas en sistema EPPO).

---
## Bloque 1: Instalación de Librerías 📚

In [ ]:
# Instalar PlantCV para análisis de imágenes de plantas
!pip install plantcv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.5/60.5 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 2.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.3/279.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.1/102.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.5 MB/s eta 0

## Bloque 2: Importar Módulos 🔧

In [ ]:
# ── Librerías estándar ──────────────────────────────────────────────────────
import os
import json
import glob
from collections import Counter

# ── Manipulación de datos ───────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualización ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import ListedColormap
import plotly.express as px
import plotly.io

# ── Procesamiento de imágenes (scikit-image) ───────────────────────────────
from skimage.io import imread
from skimage.color import rgb2hsv, rgb2lab, rgb2gray
from skimage.transform import resize
from skimage.filters import threshold_otsu, gaussian
from skimage.exposure import histogram
from skimage import morphology, segmentation
import skimage as ski

# ── Operaciones morfológicas (scipy) ───────────────────────────────────────
from scipy import ndimage as ndi

# ── PlantCV ────────────────────────────────────────────────────────────────
from plantcv import plantcv as pcv

# ── Machine Learning (sklearn) ─────────────────────────────────────────────
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

print(" Todas las librerías importadas correctamente")
print(f"PlantCV versión: {pcv.__version__}")

## Bloque 3: Montar Google Drive y Configurar Rutas

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
# ── Rutas del dataset ──────────────────────────────────────────────────────
# INSTRUCCIÓN: Ajuste estas rutas según la ubicación del dataset en su Drive
RUTA_BASE    = "/content/gdrive/MyDrive/BigData & Bioinformática/Trabajo final/dataset"  # Carpeta raíz del dataset
RUTA_IMG     = os.path.join(RUTA_BASE, "img")      # Carpeta img/
RUTA_ANN     = os.path.join(RUTA_BASE, "ann")      # Carpeta ann/
RUTA_META    = os.path.join(RUTA_BASE, "meta.json")
RUTA_OUTPUT  = os.path.join(RUTA_BASE, "resultados")  # Carpeta para guardar resultados

os.makedirs(RUTA_OUTPUT, exist_ok=True)

todas_imagenes = sorted(glob.glob(os.path.join(RUTA_IMG, "*.jpg")))

# Listar anotaciones
anotaciones = sorted(glob.glob(os.path.join(RUTA_ANN, "*.json")))

print(f" Total imágenes:    {len(todas_imagenes)}")
print(f" Anotaciones JSON:  {len(anotaciones)}")

 Total imágenes:    97
 Anotaciones JSON:  97


## Bloque 4: Configurar PlantCV

In [ ]:
# Configuración de parámetros del flujo de trabajo de PlantCV
pcv.params.debug          = "plot"   # "plot" para visualizar, "print" para guardar, None para silenciar
pcv.params.dpi            = 100
pcv.params.text_size      = 10
pcv.params.text_thickness = 5
pcv.params.line_thickness = 3

---
## Bloque 5: Exploración del Dataset — Metadata y Estadísticas

In [ ]:
# ── Leer meta.json ─────────────────────────────────────────────────────────
with open(RUTA_META) as f:
    meta = json.load(f)

print(f"Total de clases definidas en meta.json: {len(meta['classes'])}")
print(f"Tipo de proyecto: {meta['projectType']}")
print("\nPrimeras 5 clases:")
for cls in meta['classes'][:5]:
    print(f"  Código EPPO: {cls['title']} | Color: {cls['color']}")

Total de clases definidas en meta.json: 52
Tipo de proyecto: images

Primeras 5 clases:
  Código EPPO: 1COMF | Color: #0F8A3D
  Código EPPO: 1PLAK | Color: #3D0F8A
  Código EPPO: 3UNCLK | Color: #0F8A2F
  Código EPPO: ALOMY | Color: #8A310F
  Código EPPO: ANGAR | Color: #8A0F53


In [ ]:
# ── Función auxiliar: parsear una anotación JSON ───────────────────────────
def parsear_anotacion(ruta_json):
    """Parsea un archivo JSON de anotación y retorna un diccionario con los datos clave."""
    with open(ruta_json) as f:
        data = json.load(f)
    tags = {t['name']: t['value'] for t in data.get('tags', [])}
    objetos = []
    for obj in data.get('objects', []):
        pts = obj['points']['exterior']
        x1, y1 = pts[0]
        x2, y2 = pts[1]
        objetos.append({
            'clase'   : obj['classTitle'],
            'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
            'ancho'   : abs(x2 - x1),
            'alto'    : abs(y2 - y1),
            'area_px' : abs(x2 - x1) * abs(y2 - y1)
        })
    return {
        'tags'    : tags,
        'objetos' : objetos,
        'size'    : data.get('size', {})
    }

# ── Construir DataFrame global con todas las anotaciones ──────────────────
registros = []
eppo_map  = {}  # Diccionario EPPO → {latin, english}

for ruta_json in anotaciones:
    nombre_base = os.path.basename(ruta_json).replace('.json', '')
    ann = parsear_anotacion(ruta_json)
    tags = ann['tags']

    # Guardar mapeo EPPO
    eppo = tags.get('eppo', '?')
    if eppo not in eppo_map:
        eppo_map[eppo] = {
            'latin'  : tags.get('latin', '?'),
            'english': tags.get('english', '?')
        }

    for obj in ann['objetos']:
        registros.append({
            'imagen'    : nombre_base,
            'trial_id'  : tags.get('trial_id', '?'),
            'box_id'    : tags.get('box_id', '?'),
            'eppo_img'  : eppo,        # especie principal de la caja
            'fecha'     : tags.get('date', '?')[:10],
            'clase_obj' : obj['clase'],
            'x1': obj['x1'], 'y1': obj['y1'],
            'x2': obj['x2'], 'y2': obj['y2'],
            'ancho'     : obj['ancho'],
            'alto'      : obj['alto'],
            'area_px'   : obj['area_px']
        })

df = pd.DataFrame(registros)
print(f"DataFrame construido: {df.shape[0]} filas × {df.shape[1]} columnas")
df.head(10)

OSError: [Errno 107] Transport endpoint is not connected

In [ ]:
# ── Estadísticas generales ─────────────────────────────────────────────────
print("=" * 55)
print("       ESTADÍSTICAS GENERALES DEL DATASET (subconjunto)")
print("=" * 55)
print(f"  Total objetos anotados : {len(df):,}")
print(f"  Imágenes únicas        : {df['imagen'].nunique()}")
print(f"  Clases únicas          : {df['clase_obj'].nunique()}")
print(f"  Trials                 : {sorted(df['trial_id'].unique())}")
print(f"  Período                : {df['fecha'].min()} → {df['fecha'].max()}")
print()

objs_por_imagen = df.groupby('imagen').size()
print(f"  Objetos por imagen:")
print(f"    Mínimo   = {objs_por_imagen.min()}")
print(f"    Máximo   = {objs_por_imagen.max()}")
print(f"    Promedio = {objs_por_imagen.mean():.1f}")
print(f"    Mediana  = {objs_por_imagen.median():.0f}")
print()

print("  Área promedio de bounding boxes (px²):")
print(f"    Promedio = {df['area_px'].mean():,.0f}")
print(f"    Mínimo   = {df['area_px'].min():,}")
print(f"    Máximo   = {df['area_px'].max():,}")

In [ ]:
# ── Distribución de clases (Top 15) ───────────────────────────────────────
conteo_clases = df['clase_obj'].value_counts().head(15)

# Añadir nombre en inglés (cuando esté disponible en eppo_map)
etiquetas = []
for eppo in conteo_clases.index:
    eng = eppo_map.get(eppo, {}).get('english', '')
    etiquetas.append(f"{eppo}\n({eng})" if eng else eppo)

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(range(len(conteo_clases)), conteo_clases.values, color=plt.cm.tab20.colors[:len(conteo_clases)])
ax.set_xticks(range(len(conteo_clases)))
ax.set_xticklabels(etiquetas, fontsize=9)
ax.set_xlabel("Especie (Código EPPO)")
ax.set_ylabel("Número de bounding boxes")
ax.set_title("Top 15 clases de malezas — Subconjunto OPPD")
for bar, val in zip(bars, conteo_clases.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(val),
            ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_OUTPUT, "distribucion_clases.png"), dpi=150)
plt.show()

In [ ]:
# ── Distribución de imágenes por Trial ────────────────────────────────────
imgs_por_trial = df.groupby('trial_id')['imagen'].nunique()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico 1: Imágenes por trial
axes[0].pie(imgs_por_trial.values, labels=imgs_por_trial.index,
            autopct='%1.1f%%', colors=plt.cm.Paired.colors, startangle=90)
axes[0].set_title("Imágenes por Trial")

# Gráfico 2: Objetos por trial
objs_trial = df.groupby('trial_id').size()
axes[1].bar(objs_trial.index, objs_trial.values, color=plt.cm.Paired.colors[:4])
axes[1].set_xlabel("Trial")
axes[1].set_ylabel("Número de bounding boxes")
axes[1].set_title("Objetos anotados por Trial")

plt.tight_layout()
plt.savefig(os.path.join(RUTA_OUTPUT, "distribucion_trials.png"), dpi=150)
plt.show()

---
## Bloque 6: Lectura y Visualización de Imágenes con Anotaciones

In [ ]:
# ── Función: buscar imagen correspondiente a una anotación ────────────────
def buscar_imagen(nombre_base):
    """Busca el archivo JPG en la carpeta img/"""
    ruta = os.path.join(RUTA_IMG, nombre_base)
    return ruta if os.path.exists(ruta) else None

# ── Seleccionar una imagen de ejemplo ─────────────────────────────────────
ann_ejemplo = anotaciones[5]  # Puede cambiar el índice
nombre_base = os.path.basename(ann_ejemplo).replace('.json', '')
ruta_img_ejemplo = buscar_imagen(nombre_base)

print(f"Anotación seleccionada: {nombre_base}")
print(f"Imagen encontrada: {ruta_img_ejemplo}")

# Parsear anotación
ann_data = parsear_anotacion(ann_ejemplo)
tags_img = ann_data['tags']
print(f"\nEspecie principal (EPPO) : {tags_img.get('eppo','?')}")
print(f"Nombre latino            : {tags_img.get('latin','?')}")
print(f"Nombre común             : {tags_img.get('english','?')}")
print(f"Fecha de captura         : {tags_img.get('date','?')[:10]}")
print(f"Trial ID                 : {tags_img.get('trial_id','?')}")
print(f"Objetos anotados         : {len(ann_data['objetos'])}")

In [ ]:
# ── Leer imagen con PlantCV ────────────────────────────────────────────────
pcv.params.debug = None  # Silenciar debug para esta celda
color_img, path, filename = pcv.readimage(filename=ruta_img_ejemplo)

print(f"Shape imagen (BGR): {color_img.shape}")
print(f"Tipo de dato       : {color_img.dtype}")
print(f"Valores min-max    : {color_img.min()} — {color_img.max()}")

In [ ]:
# ── Visualizar imagen RGB con bounding boxes anotados ─────────────────────
# PlantCV carga en BGR → convertir a RGB para matplotlib
img_rgb = color_img[..., [2, 1, 0]]

# Asignar un color único a cada clase presente
clases_presentes = list(set(o['clase'] for o in ann_data['objetos']))
colores_bbox = plt.cm.tab20.colors
color_por_clase = {cls: colores_bbox[i % 20] for i, cls in enumerate(clases_presentes)}

fig, ax = plt.subplots(1, 1, figsize=(16, 10))
ax.imshow(img_rgb)

for obj in ann_data['objetos']:
    x1, y1, x2, y2 = obj['x1'], obj['y1'], obj['x2'], obj['y2']
    color = color_por_clase[obj['clase']]
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                               linewidth=1.5, edgecolor=color, facecolor='none')
    ax.add_patch(rect)

# Leyenda
from matplotlib.lines import Line2D
leyenda = [Line2D([0], [0], color=color_por_clase[c], lw=2, label=c)
           for c in clases_presentes]
ax.legend(handles=leyenda, loc='upper right', fontsize=8)
ax.set_title(f"Imagen con bounding boxes — {tags_img.get('english','?')} ({tags_img.get('eppo','?')}) | {len(ann_data['objetos'])} objetos")
ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(RUTA_OUTPUT, "imagen_con_bboxes.png"), dpi=120)
plt.show()

In [ ]:
# ── Visualización interactiva con Plotly (para inspección de píxeles) ─────
fig_plotly = px.imshow(img_rgb, title=f"Visualización interactiva — {tags_img.get('latin','?')}")
plotly.io.show(fig_plotly)

---
## Bloque 7: Análisis de Espacios de Color

In [ ]:
# ── Recortar una región de trabajo para análisis de color ─────────────────
# Usamos la mitad central de la imagen para evitar bordes
filas, cols = img_rgb.shape[:2]
crop_img = img_rgb[filas//4 : 3*filas//4,
                   cols//4 : 3*cols//4, :]

print(f"Shape imagen completa : {img_rgb.shape}")
print(f"Shape imagen recortada: {crop_img.shape}")
plt.imshow(crop_img)
plt.title("Región de trabajo (crop central)")
plt.axis('off')
plt.show()

In [ ]:
# ── Transformación a diferentes espacios de color ─────────────────────────
# Normalizar a float [0,1] para skimage
crop_float = crop_img.astype("float32") / 255.0

# Convertir espacios
hsv_img = rgb2hsv(crop_float)
lab_img = rgb2lab(crop_float)
gray_img = rgb2gray(crop_float)

# Bandas RGB
r = crop_float[:, :, 0]
g = crop_float[:, :, 1]
b = crop_float[:, :, 2]

# Bandas HSV
h = hsv_img[:, :, 0]
s = hsv_img[:, :, 1]
v = hsv_img[:, :, 2]

# Bandas LAB
L_band = lab_img[:, :, 0]
A_band = lab_img[:, :, 1]
B_band = lab_img[:, :, 2]

print("Rango HSV por canal:")
for nombre, banda in zip(['H','S','V'], [h, s, v]):
    print(f"  {nombre}: [{banda.min():.3f}, {banda.max():.3f}]")
print("Rango LAB por canal:")
for nombre, banda in zip(['L','A','B'], [L_band, A_band, B_band]):
    print(f"  {nombre}: [{banda.min():.2f}, {banda.max():.2f}]")

In [ ]:
# ── Comparación visual de espacios de color ───────────────────────────────
plot_rows, plot_cols = 3, 4
fig, ax = plt.subplots(plot_rows, plot_cols, figsize=(plot_cols * 4, plot_rows * 3))

# RGB
ax[0,0].imshow(crop_img); ax[0,0].set_title("Imagen RGB")
ax[0,1].imshow(r, cmap='Reds');   ax[0,1].set_title("Canal Rojo (R)")
ax[0,2].imshow(g, cmap='Greens'); ax[0,2].set_title("Canal Verde (G)")
ax[0,3].imshow(b, cmap='Blues');  ax[0,3].set_title("Canal Azul (B)")

# HSV
ax[1,0].imshow(hsv_img); ax[1,0].set_title("Imagen HSV")
ax[1,1].imshow(h, cmap='hsv');  ax[1,1].set_title("Matiz (H)")
ax[1,2].imshow(s, cmap='gray'); ax[1,2].set_title("Saturación (S)")
ax[1,3].imshow(v, cmap='gray'); ax[1,3].set_title("Valor/Brillo (V)")

# LAB
ax[2,0].imshow(crop_img); ax[2,0].set_title("Referencia RGB")
ax[2,1].imshow(L_band, cmap='gray'); ax[2,1].set_title("Luminancia (L)")
ax[2,2].imshow(A_band, cmap='RdYlGn_r'); ax[2,2].set_title("Verde→Rojo (a*)")
ax[2,3].imshow(B_band, cmap='coolwarm'); ax[2,3].set_title("Azul→Amarillo (b*)")

for row in ax:
    for a in row:
        a.axis('off')

plt.suptitle("Comparación de Espacios de Color — Imagen de Malezas", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_OUTPUT, "espacios_color.png"), dpi=150, bbox_inches='tight')
plt.show()

---
## Bloque 8: Histogramas y Selección de Canal para Segmentación

In [ ]:
from skimage.exposure import histogram as skimage_hist

canales = {
    "Verde RGB (G)" : g,
    "Saturación HSV (H)" : h,
    "Verde→Rojo LAB (a*)" : A_band,
    "Escala de grises" : gray_img
}

fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for i, (nombre, banda) in enumerate(canales.items()):
    # Imagen del canal
    axes[0, i].imshow(banda, cmap='gray')
    axes[0, i].set_title(nombre)
    axes[0, i].axis('off')

    # Histograma
    freq, val = skimage_hist(banda)
    axes[1, i].plot(val, freq, lw=1.5, color='steelblue')
    axes[1, i].axvline(np.mean(banda), color='red', linestyle='--', label=f'Media={np.mean(banda):.2f}') #ESCOGER LIMITE OTSU
    axes[1, i].set_title(f"Histograma — {nombre}")
    axes[1, i].set_xlabel("Intensidad")
    axes[1, i].set_ylabel("Frecuencia")
    axes[1, i].legend(fontsize=8)

plt.suptitle("Histogramas de canales candidatos para segmentación", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_OUTPUT, "histogramas_canales.png"), dpi=150)
plt.show()

---
## Bloque 9:  SEGMENTACIÓN — Separación Planta vs. Suelo

La segmentación es el paso crítico: separar los **píxeles de vegetación (maleza)** del **suelo y fondo**. Se comparan tres enfoques vistos en clase.

In [ ]:
# ── Método 1: Umbral manual sobre canal Verde RGB ─────────────────────────
# El canal verde tiene valores más altos en zonas con vegetación
umbral_manual = 0.5  # Ajuste según histograma
mascara_manual = np.where(g > umbral_manual, 1, 0).astype("uint8")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(g, cmap='gray'); axes[0].set_title("Canal Verde (G)")
axes[1].plot(*skimage_hist(g)[::-1]);
axes[1].axvline(umbral_manual, color='red', linestyle='--', label=f'Umbral = {umbral_manual}')
axes[1].set_title("Histograma + Umbral Manual")
axes[1].set_xlabel("Intensidad"); axes[1].set_ylabel("Frecuencia")
axes[1].legend()
axes[2].imshow(mascara_manual, cmap='gray'); axes[2].set_title("Máscara — Umbral Manual")
for a in [axes[0], axes[2]]: a.axis('off')
plt.suptitle("Segmentación: Umbral Manual — Canal Verde", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Método 2: Umbral automático de Otsu ───────────────────────────────────
from skimage.filters import threshold_otsu

umbral_otsu_h = threshold_otsu(h)
mascara_otsu = (h > umbral_otsu_h).astype("uint8")

print(f"Umbral Otsu (canal H): {umbral_otsu_h:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(h, cmap='gray'); axes[0].set_title("Canal H")
freq, val = skimage_hist(h)
axes[1].plot(val, freq, lw=1.5, color='steelblue')
axes[1].axvline(umbral_otsu_h, color='red', linestyle='--', label=f'Otsu = {umbral_otsu_h:.3f}')
axes[1].set_title("Histograma + Umbral Otsu")
axes[1].set_xlabel("Intensidad"); axes[1].set_ylabel("Frecuencia")
axes[1].legend()
axes[2].imshow(mascara_otsu, cmap='gray'); axes[2].set_title(f"Máscara Otsu (umbral={umbral_otsu_h:.3f})")
for a in [axes[0], axes[2]]: a.axis('off')
plt.suptitle("Segmentación: Método de Otsu — Canal H", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_OUTPUT, "segmentacion_otsu.png"), dpi=150)
plt.show()

In [ ]:
# ── Método 3: Canal LAB (a*) — mejor separación planta/suelo ──────────────
# En el espacio LAB, valores negativos de a* = verde, positivos = rojo/suelo
# El suelo tiene a* más alto (rojo-anaranjado)
# Las plantas tienen a* más bajo (verde)

umbral_lab = threshold_otsu(A_band)
# Los píxeles de planta tienen a* MENOR que el umbral (más verdes)
mascara_lab = (A_band < umbral_lab).astype("uint8")

print(f"Umbral Otsu (canal LAB a*): {umbral_lab:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(A_band, cmap='RdYlGn_r'); axes[0].set_title("Canal LAB a* (verde→rojo)")
freq_lab, val_lab = skimage_hist(A_band)
axes[1].plot(val_lab, freq_lab, lw=1.5, color='steelblue')
axes[1].axvline(umbral_lab, color='red', linestyle='--', label=f'Otsu a* = {umbral_lab:.2f}')
axes[1].set_title("Histograma LAB a* + Umbral Otsu")
axes[1].set_xlabel("Valor a*"); axes[1].set_ylabel("Frecuencia")
axes[1].legend()
axes[2].imshow(mascara_lab, cmap='gray'); axes[2].set_title("Máscara — LAB a* (Otsu)")
for a in [axes[0], axes[2]]: a.axis('off')
plt.suptitle("Segmentación: Canal LAB a* — Mejor discriminación planta/suelo", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_OUTPUT, "segmentacion_lab.png"), dpi=150)
plt.show()

In [ ]:
# ── Comparación de los 3 métodos de segmentación ─────────────────────────
pcv.params.debug = None

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

axes[0,0].imshow(crop_img); axes[0,0].set_title("Imagen RGB original", fontweight='bold')
axes[0,1].imshow(mascara_manual, cmap='gray'); axes[0,1].set_title(f"Método 1: Umbral Manual\nCanal Verde (G>{umbral_manual:.2f})")
axes[0,2].imshow(mascara_otsu, cmap='gray'); axes[0,2].set_title(f"Método 2: Otsu automático\nCanal H (H>{umbral_otsu_h:.3f})")
axes[1,0].imshow(mascara_lab, cmap='gray'); axes[1,0].set_title(f"Método 3: Otsu LAB a*\n(a*<{umbral_lab:.2f})")

# Superponer máscara LAB sobre imagen original
overlay = crop_img.copy()
overlay[mascara_lab == 0] = [180, 90, 30]  # Colorear fondo como marrón
axes[1,1].imshow(overlay); axes[1,1].set_title("Overlay — Método 3 sobre imagen RGB")

axes[1,2].axis('off') # Turn off the unused subplot

for row in axes:
    for a in row: a.axis('off')

plt.suptitle("Comparación de Métodos de Segmentación — Malezas OPPD", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_OUTPUT, "comparacion_segmentacion.png"), dpi=150)
plt.show()

# Estadísticas de cobertura
total_px = mascara_lab.size
for nombre, mascara in [("Manual", mascara_manual), ("Otsu G", mascara_otsu),
                         ("Otsu LAB a*", mascara_lab)]:
    cobertura = 100 * mascara.sum() / total_px
    print(f"  {nombre:15s}: {cobertura:.1f}% de píxeles clasificados como planta")

---
## Bloque 10: Operaciones Morfológicas

In [ ]:
# ── Seleccionar la mejor máscara (LAB a*) para aplicar morfología ─────────
mascara_base = mascara_lab.copy()

# Elemento estructurante para las operaciones
footprint_3 = np.ones((3, 3), dtype='uint8')   # 3x3
footprint_7 = np.ones((7, 7), dtype='uint8')   # 7x7 (más agresivo)

# ── Operaciones morfológicas básicas ──────────────────────────────────────
erosion  = ski.morphology.erosion(mascara_base, footprint_3).astype('uint8')
dilatacion = ski.morphology.dilation(mascara_base, footprint_3).astype('uint8')
apertura = ski.morphology.opening(mascara_base, footprint_3).astype('uint8')    # erosión → dilatación
cierre   = ski.morphology.closing(mascara_base, footprint_3).astype('uint8')    # dilatación → erosión
relleno  = ndi.binary_fill_holes(mascara_base).astype('uint8')                  # Rellenar huecos

# ── Visualización de las operaciones morfológicas ─────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

pares = [
    (mascara_base, "Máscara Original"),
    (erosion,      "Erosión (3×3)\nElimina objetos pequeños"),
    (dilatacion,   "Dilatación (3×3)\nExpande objetos"),
    (apertura,     "Apertura (3×3)\nErosión + Dilatación"),
    (cierre,       "Cierre (3×3)\nDilatación + Erosión"),
    (relleno,      "Rellenar Huecos\n(scipy fill_holes)")
]

for ax, (mask, titulo) in zip(axes.flat, pares):
    ax.imshow(mask, cmap='gray')
    ax.set_title(titulo)
    ax.axis('off')

plt.suptitle("Operaciones Morfológicas sobre Máscara de Vegetación", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_OUTPUT, "operaciones_morfologicas.png"), dpi=150)
plt.show()

In [ ]:
# ── Pipeline morfológico completo para limpieza de máscara ────────────────
# Paso 1: Cierre para unir fragmentos cercanos
mascara_cerrada = ski.morphology.closing(mascara_base, footprint_7).astype('uint8')

# Paso 2: Rellenar huecos internos - (EXPLICAR EL RELLENO POR SOBRELAPE)
mascara_rellena = ndi.binary_fill_holes(mascara_cerrada).astype('uint8')

# Paso 3: Eliminar objetos pequeños (ruido) — usando PlantCV
pcv.params.debug = None
# Convertir a 8 bits para PlantCV (0 o 255)
mascara_pcv_input = (mascara_rellena * 255).astype('uint8')
mascara_limpia = pcv.fill(bin_img=mascara_pcv_input, size=200)   # Eliminar grupos < 200 px

# Comparación: original vs. pipeline limpio
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(crop_img); axes[0].set_title("Imagen RGB"); axes[0].axis('off')
axes[1].imshow(mascara_base, cmap='gray'); axes[1].set_title("Máscara inicial\n(Otsu LAB a*)"); axes[1].axis('off')
axes[2].imshow(mascara_rellena, cmap='gray'); axes[2].set_title("Tras cierre + relleno"); axes[2].axis('off')
axes[3].imshow(mascara_limpia, cmap='gray'); axes[3].set_title("Tras pcv.fill()\n(sin ruido pequeño)"); axes[3].axis('off')

plt.suptitle("Pipeline Morfológico de Limpieza de Máscara", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_OUTPUT, "pipeline_morfologico.png"), dpi=150)
plt.show()

# Estadísticas de mejora
print(f"Píxeles planta inicial  : {mascara_base.sum():,}")
print(f"Píxeles planta limpia   : {(mascara_limpia > 0).sum():,}")
print(f"Diferencia              : {(mascara_limpia > 0).sum() - mascara_base.sum():,} px")

---
## Bloque 11: Filtros y Mejoramiento de Imagen (Kernels/Convoluciones)

In [ ]:
from skimage.filters import sobel, prewitt, gaussian as gauss_filter
from skimage.feature import canny

# Aplicar filtros sobre la imagen LAB con la banda A
lab_uint8 = (mascara_lab * 255).astype('uint8')

bordes_sobel   = sobel(mascara_lab)
bordes_prewitt = prewitt(mascara_lab)
bordes_canny   = canny(A_band, sigma=2.0)
img_suavizada  = gauss_filter(mascara_lab, sigma=2.0)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes[0,0].imshow(mascara_lab, cmap='gray');        axes[0,0].set_title("Máscara LAB")
axes[0,1].imshow(img_suavizada, cmap='gray');   axes[0,1].set_title("Filtro Gaussiano (σ=2)\nSuavizado / reducción ruido")
axes[0,2].imshow(bordes_sobel, cmap='gray');    axes[0,2].set_title("Filtro Sobel\nDetección de bordes (gradiente)")
axes[1,0].imshow(bordes_prewitt, cmap='gray');  axes[1,0].set_title("Filtro Prewitt\nDetección de bordes")
axes[1,1].imshow(bordes_canny, cmap='gray');    axes[1,1].set_title("Detector Canny (σ=2)\nBordes finos binarios")

# Superponer bordes Canny sobre la imagen RGB
overlay_canny = crop_img.copy()
overlay_canny[bordes_canny] = [255, 0, 0]  # Bordes en rojo
axes[1,2].imshow(overlay_canny);              axes[1,2].set_title("Bordes Canny sobre RGB")

for row in axes:
    for a in row: a.axis('off')

plt.suptitle("Filtros y Convoluciones — Detección de Bordes", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_OUTPUT, "filtros_convoluciones.png"), dpi=150)
plt.show()

---
## Bloque 12: Análisis con PlantCV — Extracción de Características

In [ ]:
# ── Análisis de tamaño y forma con PlantCV ────────────────────────────────
pcv.params.debug = "plot"

# Asegurar que los arrays sean contiguos en memoria (fix para OpenCV) #EXPLICAR PORQUE LO HICIMOS ASÍ Y NO CON CADA UNO DE LOS BOUNDING BOX
# Crop the original BGR image to match the region where masks were created
cropped_bgr_img = color_img[filas//4 : 3*filas//4, cols//4 : 3*cols//4, :]
crop_bgr_cont = np.ascontiguousarray(cropped_bgr_img)
mascara_cont    = np.ascontiguousarray(mascara_limpia)

# Análisis de tamaño (área, perímetro, etc.)
shape_img = pcv.analyze.size(img=crop_bgr_cont, labeled_mask=mascara_cont, label="planta")

# Mostrar métricas calculadas
print("\n📊 Métricas de forma calculadas por PlantCV:")
for key, val in pcv.outputs.observations.get('planta', {}).items():
    print(f"  {key:30s}: {val['value']}")

---
## Bloque 13: Extracción de Recortes (Crops) de Bounding Boxes

In [ ]:
# ── Función: extraer recortes individuales de cada planta ─────────────────
def extraer_crops(img_bgr, objetos, tam_objetivo=(64, 64)):
    """
    Extrae recortes de imagen para cada bounding box anotado.
    Retorna lista de (crop_rgb, clase, area_px).
    """
    img_rgb_full = img_bgr[..., [2, 1, 0]]  # BGR → RGB
    h_img, w_img = img_rgb_full.shape[:2]
    crops = []
    for obj in objetos:
        x1 = max(0, min(obj['x1'], obj['x2']))
        y1 = max(0, min(obj['y1'], obj['y2']))
        x2 = min(w_img, max(obj['x1'], obj['x2']))
        y2 = min(h_img, max(obj['y1'], obj['y2']))
        if x2 - x1 < 5 or y2 - y1 < 5:
            continue  # Ignorar bboxes demasiado pequeñas
        crop = img_rgb_full[y1:y2, x1:x2]
        crop_resized = resize(crop, tam_objetivo, anti_aliasing=True)
        crops.append((crop_resized, obj['clase'], obj['area_px']))
    return crops

# Extraer crops de la imagen de ejemplo
pcv.params.debug = None
color_img_full, _, _ = pcv.readimage(filename=ruta_img_ejemplo)
crops_ejemplo = extraer_crops(color_img_full, ann_data['objetos'])

print(f"Total de crops extraídos: {len(crops_ejemplo)}")

# Mostrar una muestra de crops
n_mostrar = min(12, len(crops_ejemplo))
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for i, ax in enumerate(axes.flat):
    if i < n_mostrar:
        crop, clase, area = crops_ejemplo[i]
        ax.imshow(crop)
        ax.set_title(f"{clase}\n{area:,} px²", fontsize=9)
    ax.axis('off')
plt.suptitle("Recortes individuales de plantas — desde bounding boxes", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RUTA_OUTPUT, "crops_plantas.png"), dpi=150)
plt.show()

---
## Bloque 14: Análisis de Múltiples Imágenes y Generación de Dataset de Características

In [ ]:
import random

# ── Extraer características de color por bounding box en múltiples imágenes ─
# Se procesarán las N imágenes aleatorias para no saturar la memoria
N_IMAGENES = 15   # Ajustar según recursos disponibles

registros_ml = []

# Shuffle and select N_IMAGENES random annotations
selected_anotaciones = random.sample(anotaciones, min(N_IMAGENES, len(anotaciones)))

for i, ruta_json in enumerate(selected_anotaciones):
    nombre_base = os.path.basename(ruta_json).replace('.json', '')
    ruta_img = buscar_imagen(nombre_base)
    if ruta_img is None:
        print(f"    Imagen no encontrada: {nombre_base}")
        continue

    ann = parsear_anotacion(ruta_json)
    pcv.params.debug = None
    img_bgr, _, _ = pcv.readimage(filename=ruta_img)
    img_rgb_full = img_bgr[..., [2, 1, 0]]
    h_img, w_img = img_rgb_full.shape[:2]

    for obj in ann['objetos']:
        x1 = max(0, min(obj['x1'], obj['x2']))
        y1 = max(0, min(obj['y1'], obj['y2']))
        x2 = min(w_img, max(obj['x1'], obj['x2']))
        y2 = min(h_img, max(obj['y1'], obj['y2']))
        if x2 - x1 < 8 or y2 - y1 < 8:
            continue

        crop = img_rgb_full[y1:y2, x1:x2].astype('float32') / 255.0
        crop_hsv = rgb2hsv(crop)
        crop_lab = rgb2lab(crop)

        # Características: medias y desviaciones estándar de cada canal
        registros_ml.append({
            'clase'   : obj['clase'],
            'area_px' : obj['area_px'],
            # RGB
            'r_mean': crop[:,:,0].mean(), 'r_std': crop[:,:,0].std(),
            'g_mean': crop[:,:,1].mean(), 'g_std': crop[:,:,1].std(),
            'b_mean': crop[:,:,2].mean(), 'b_std': crop[:,:,2].std(),
            # HSV
            'h_mean': crop_hsv[:,:,0].mean(), 'h_std': crop_hsv[:,:,0].std(),
            's_mean': crop_hsv[:,:,1].mean(), 's_std': crop_hsv[:,:,1].std(),
            'v_mean': crop_hsv[:,:,2].mean(), 'v_std': crop_hsv[:,:,2].std(),
            # LAB
            'L_mean': crop_lab[:,:,0].mean(), 'L_std': crop_lab[:,:,0].std(),
            'a_mean': crop_lab[:,:,1].mean(), 'a_std': crop_lab[:,:,1].std(),
            'b_mean': crop_lab[:,:,2].mean(), 'b_std': crop_lab[:,:,2].std(),
        })

    print(f"  [{i+1:2d}/{N_IMAGENES}] {nombre_base} — {len(ann['objetos'])} objetos procesados")

df_ml = pd.DataFrame(registros_ml)
print(f"\n Dataset de características: {df_ml.shape[0]} muestras × {df_ml.shape[1]} columnas")
df_ml.head()

In [ ]:
# ── Filtrar clases con al menos 20 muestras ──────────────────────────────
conteo_ml = df_ml['clase'].value_counts()
clases_validas = conteo_ml[conteo_ml >= 20].index.tolist()
df_ml_filtrado = df_ml[df_ml['clase'].isin(clases_validas)].copy()

print(f"Clases con ≥20 muestras: {len(clases_validas)}")
print(f"Muestras totales para ML: {len(df_ml_filtrado)}")
print("\nDistribución:")
for cls in clases_validas:
    eng = eppo_map.get(cls, {}).get('english', '')
    print(f"  {cls:8s} ({eng:30s}): {conteo_ml[cls]} muestras")

---
## Bloque 15: Machine Learning — Clasificación de Especies

Se aplica el clasificador **Naive Bayes Gaussiano** para clasificar especies de malezas a partir de las características de color extraídas de cada bounding box.

In [ ]:
# ── Preparar datos de entrenamiento ───────────────────────────────────────
# Columnas de características (color stats en RGB, HSV y LAB)
columnas_X = ['r_mean','r_std','g_mean','g_std','b_mean','b_std',
               'h_mean','h_std','s_mean','s_std','v_mean','v_std',
               'L_mean','L_std','a_mean','a_std','b_mean','b_std']

X = df_ml_filtrado[columnas_X].values
y_str = df_ml_filtrado['clase'].values

# Codificar etiquetas a enteros
le = LabelEncoder()
y = le.fit_transform(y_str)

print("Codificación de clases:")
for codigo, clase in enumerate(le.classes_):
    eng = eppo_map.get(clase, {}).get('english', '?')
    print(f"  Código {codigo}: {clase} ({eng})")

print(f"\nX.shape = {X.shape}")
print(f"y.shape = {y.shape}")

In [ ]:
# ── División entrenamiento / prueba (80% / 20%) ───────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Conjuntos de datos:")
for nombre, arr in [("X_train",X_train),("X_test",X_test),("y_train",y_train),("y_test",y_test)]:
    print(f"  {nombre}: {arr.shape}")

In [ ]:
# ── Entrenar modelo Naive Bayes Gaussiano ─────────────────────────────────
modelo = GaussianNB()
modelo.fit(X_train, y_train)
print(" Modelo entrenado exitosamente")

# Predicciones
y_pred = modelo.predict(X_test)

# Evaluación
accuracy = modelo.score(X_test, y_test)
print(f"\n Accuracy en conjunto de prueba: {accuracy * 100:.2f}%")

In [ ]:
# ── Reporte de clasificación ───────────────────────────────────────────────
etiquetas_clases = [f"{cls}" for cls in le.classes_]
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=etiquetas_clases))

In [ ]:
# ── Matriz de confusión ────────────────────────────────────────────────────
matriz = confusion_matrix(y_test, y_pred, labels=np.unique(y_test))
etiquetas_presentes = le.classes_[np.unique(y_test)]

disp = ConfusionMatrixDisplay(confusion_matrix=matriz, display_labels=etiquetas_presentes)
fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(ax=ax, cmap='Blues', colorbar=True)
plt.title(f"Matriz de Confusión — Naive Bayes Gaussiano\nAccuracy: {accuracy*100:.2f}%", fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(RUTA_OUTPUT, "matriz_confusion.png"), dpi=150)
plt.show()

In [ ]:
# ── Guardar dataset de características en CSV ─────────────────────────────
ruta_csv = os.path.join(RUTA_OUTPUT, "features_malezas.csv")
df_ml_filtrado.to_csv(ruta_csv, index=False)
print(f" Dataset de características guardado en: {ruta_csv}")

---
## Bloque 16: Guardado de Resultados

In [ ]:
# ── Resumen final y guardado ──────────────────────────────────────────────

# Guardar DataFrame completo de anotaciones
df.to_csv(os.path.join(RUTA_OUTPUT, "anotaciones_completas.csv"), index=False)

# Guardar resumen de clases
resumen_clases = df.groupby('clase_obj').agg(
    conteo       = ('clase_obj', 'count'),
    area_media   = ('area_px', 'mean'),
    area_min     = ('area_px', 'min'),
    area_max     = ('area_px', 'max'),
).reset_index()
resumen_clases['nombre_english'] = resumen_clases['clase_obj'].map(
    lambda e: eppo_map.get(e, {}).get('english', '?')
)
resumen_clases['nombre_latin'] = resumen_clases['clase_obj'].map(
    lambda e: eppo_map.get(e, {}).get('latin', '?')
)
resumen_clases = resumen_clases.sort_values('conteo', ascending=False)
resumen_clases.to_csv(os.path.join(RUTA_OUTPUT, "resumen_clases.csv"), index=False)

print(" Archivos guardados en:", RUTA_OUTPUT)
print()
print("Archivos generados:")
for f in sorted(os.listdir(RUTA_OUTPUT)):
    ruta_f = os.path.join(RUTA_OUTPUT, f)
    tam = os.path.getsize(ruta_f) / 1024
    print(f"  {f:45s} ({tam:8.1f} KB)")